# Module 6: UDFs in PySpark

Run in Google Colab or a local PySpark environment.

In [ ]:
# Run this cell first
!pip install -q pyspark

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder.appName("LearningModules").getOrCreate()
sc = spark.sparkContext

In [ ]:
# Load datasets (adjust path if needed)
students_df = spark.read.csv("../data/students.csv", header=True, inferSchema=True)
enrollments_df = spark.read.csv("../data/enrollments.csv", header=True, inferSchema=True)

## Defining a UDF with the Decorator

UDFs (User Defined Functions) let you apply custom Python logic to DataFrame columns.

In [ ]:
# Define a UDF using the decorator
@udf(returnType=StringType())
def gpa_letter_grade(gpa):
    if gpa is None:
        return "N/A"
    elif gpa >= 3.7:
        return "A"
    elif gpa >= 3.3:
        return "A-"
    elif gpa >= 3.0:
        return "B+"
    elif gpa >= 2.7:
        return "B"
    else:
        return "C or below"

# Apply the UDF
students_df.withColumn("letter_grade", gpa_letter_grade(col("gpa"))).show()

## Registering a UDF for SQL

You can also register UDFs so they can be used in SQL queries.

In [ ]:
# Register the UDF for SQL use
def classify_gpa(gpa):
    if gpa is None:
        return "Unknown"
    elif gpa >= 3.5:
        return "Honors"
    elif gpa >= 3.0:
        return "Good Standing"
    else:
        return "Needs Improvement"

spark.udf.register("classify_gpa", classify_gpa, StringType())

# Use in SQL
students_df.createOrReplaceTempView("students")
spark.sql("SELECT name, gpa, classify_gpa(gpa) AS status FROM students").show()

## Pandas UDF (Vectorized UDF)

Pandas UDFs are much faster than regular UDFs because they operate on batches using Apache Arrow.

In [ ]:
import pandas as pd
from pyspark.sql.functions import pandas_udf

# Define a Pandas UDF (vectorized)
@pandas_udf(DoubleType())
def normalize_gpa(gpa_series: pd.Series) -> pd.Series:
    """Normalize GPA to a 0-1 scale (assuming 4.0 max)."""
    return gpa_series / 4.0

# Apply the Pandas UDF
students_df.withColumn("normalized_gpa", normalize_gpa(col("gpa"))).show()

## Practice Problem 1: String UDF

Create a UDF that takes a student name and returns it in uppercase with an exclamation mark (e.g., "Alice" -> "ALICE!").

<details><summary>Hint</summary>Use the @udf decorator with StringType. In the function, use .upper() + "!".</details>

In [ ]:
# Your code here


<details><summary>Solution</summary></details>

In [ ]:
# Solution
@udf(returnType=StringType())
def shout_name(name):
    if name is None:
        return None
    return name.upper() + "!"

students_df.withColumn("shouted", shout_name(col("name"))).show()

## Practice Problem 2: SQL UDF

Register a UDF called `gpa_points` that returns the integer part of GPA multiplied by 100 (e.g., 3.8 -> 380). Use it in a SQL query.

<details><summary>Hint</summary>Register with spark.udf.register(), return IntegerType, use int(gpa * 100).</details>

In [ ]:
# Your code here


<details><summary>Solution</summary></details>

In [ ]:
# Solution
def gpa_points(gpa):
    if gpa is None:
        return 0
    return int(gpa * 100)

spark.udf.register("gpa_points", gpa_points, IntegerType())

spark.sql("SELECT name, gpa, gpa_points(gpa) AS points FROM students").show()

## Practice Problem 3: Pandas UDF

Create a Pandas UDF that computes the z-score of the GPA column (subtract mean, divide by std).

<details><summary>Hint</summary>Use @pandas_udf(DoubleType()). Inside, compute (series - series.mean()) / series.std().</details>

In [ ]:
# Your code here


<details><summary>Solution</summary></details>

In [ ]:
# Solution
@pandas_udf(DoubleType())
def gpa_zscore(gpa_series: pd.Series) -> pd.Series:
    return (gpa_series - gpa_series.mean()) / gpa_series.std()

students_df.withColumn("gpa_zscore", gpa_zscore(col("gpa"))).show()